# m45 lab · Evidence, monitoring and change control for an AI-ECG detector (Part VII capstone)

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You take the RR-interval AF detector of m37 through the life of a deployed model: plan the size of a validation
study, write a TRIPOD+AI-style report card from the model's own outputs, judge its clinical value with a decision
curve, watch it meet a new population and raise a drift alarm without any labels, and test model updates against
acceptance criteria fixed **before** the update, as a predetermined change control plan (PCCP) would.

| Part | What you do | Data |
| --- | --- | --- |
| A | Place Part VII's studies on the evidence ladder | course sources |
| B | How many patients? (eq-m45-n-sens), checked by simulation | simulation |
| C | Report card and decision curve for model v1 (eq-m45-net-benefit) | synthetic ECGs |
| D | Silent drift: a new population, a CUSUM alarm without labels (eq-m45-cusum) | synthetic ECGs |
| E | Change control: three candidate updates against pre-specified criteria | synthetic ECGs |
| F | Real recordings: train on one database, monitor on another | Challenge 2017 and PTB-XL (download) |

Synthetic ECGs come from `epml.aiecg` (m37): sinus rhythm, AF, and sinus rhythm with frequent early beats
("ectopy"), 10 s at 100 Hz. A random 5% of reference labels are flipped, to stand for an imperfect reference
standard (m35). All rates, thresholds and criteria are illustrative choices for teaching, not recommendations, and
the change-control part shows the idea of a PCCP, not any regulator's requirements. Each part ends with
**checks**; the lab runs in under a minute (plus downloads in part F).

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import aiecg as ai
from epml import evidence as ev
from epml.ml import flip_labels
from epml.retrieval import wilson_interval

FS = 100.0
OUT = Path(tempfile.mkdtemp(prefix="epml_m45_"))

def cohort(n_patients, records, rhythms, seed, flip=0.05):
    rng = np.random.default_rng(seed)
    X, lab, pid, peaks = ai.make_cohort(n_patients, records, fs=FS, duration_s=10.0, rhythms=rhythms, rng=rng)
    y = np.array([rhythms[l] == "af" for l in lab], int)
    kind = np.array([rhythms[l] for l in lab])
    y_ref = np.asarray(flip_labels(y, flip, rng), int)
    return {"F": ev.rr_feature_rows(X, peaks, FS), "G": ev.rr_feature_rows(X, peaks, FS, extra=True),
            "y": y_ref, "pid": pid, "kind": kind}

## Part A · The evidence ladder

Most AI-in-EP studies sit on the lower rungs. The list below places studies cited in Part VII by design, as their
abstracts describe them. Move up the ladder before claiming patient benefit (m35, m37).

In [ ]:
ladder = ["development and internal validation", "external validation", "prospective, non-randomized",
          "randomized trial"]
studies = [
    ("wang2026", "AF recurrence from ablation quality metrics, 14 + 2 hospitals", 1),
    ("koh2026", "meta-analysis of ML vs risk scores for AF recurrence (11 studies)", 0),
    ("ko2026", "ECG foundation model, external data set", 1),
    ("hsieh2026", "ECG foundation model under skewed training demographics", 1),
    ("noseworthy2022", "AI-ECG targeted AF screening", 2),
    ("yao2021", "AI-ECG for low ejection fraction, cluster-randomized", 3),
    ("adedinsewo2024", "AI-guided cardiomyopathy screening in pregnancy", 3),
    ("he2023", "AI vs sonographer first read of ejection fraction, blinded", 3),
    ("deisenhofer2025", "AI-guided ablation of persistent AF, double-blind", 3),
]
for rung, name in enumerate(ladder):
    print(f"{rung}  {name}")
    for key, what, r in studies:
        if r == rung:
            print(f"     [{key}] {what}")

In [ ]:
# Checks
assert all(any(r == rung for _, _, r in studies) for rung in range(len(ladder)))

## Part B · How many patients?

To estimate a sensitivity near $Se$ within $\pm d$ (95% confidence) when a share $\pi$ of patients has the
condition, Buderer (1996) gives $n = z^2 Se(1-Se)/(d^2\pi)$ (eq-m45-n-sens). Check it by simulating many studies
of that size and measuring the half-width of their Wilson intervals (m43).

In [ ]:
rng = np.random.default_rng(0)
rows = []
for se, d, prev in [(0.90, 0.05, 0.20), (0.90, 0.05, 0.05), (0.80, 0.03, 0.30)]:
    n = ev.n_for_sensitivity(se, d, prev)
    widths = []
    for _ in range(500):
        n_pos = rng.binomial(n, prev)
        lo, hi = wilson_interval(rng.binomial(n_pos, se), n_pos)
        widths.append((hi - lo) / 2)
    rows.append((se, d, prev, n, np.mean(widths)))
    print(f"Se {se:.2f}, d {d:.2f}, prevalence {prev:.2f}: n = {n:6d}; simulated mean half-width {np.mean(widths):.3f}")

In [ ]:
# Checks
for se, d, prev, n, w in rows:
    assert abs(w - d) / d < 0.1
assert abs(rows[1][3] / rows[0][3] - 4) < 0.01      # a condition 4 times rarer needs 4 times as many patients

## Part C · Report card and decision curve for model v1

Model v1 is the m37 RR detector (logistic regression on the coefficient of variation and normalized RMSSD of the RR
intervals), trained on 300 patients with sinus rhythm or AF and tested on 300 others. The report card collects the
numbers a TRIPOD+AI report needs (Collins et al. 2024): sample, discrimination with a patient-level bootstrap
interval, calibration, sensitivity and specificity with intervals, and net benefit. The narrative items (data
sources, outcome definition, missing data, intended use) are for you to write.

Net benefit (Vickers and Elkin 2006) asks whether acting on the model beats acting on everyone or no one, at
threshold probabilities $p_t$ that express how a clinician weighs false positives against missed cases
(eq-m45-net-benefit).

In [ ]:
train = cohort(300, 2, ("sinus", "af"), 1)
test = cohort(300, 2, ("sinus", "af"), 2)
v1 = ev.fit_detector(train["F"], train["y"])
card_v1 = ev.report_card("v1 (RR features), original population", test["y"], v1(test["F"]), test["pid"], 0.5,
                         rng=np.random.default_rng(0))
md_card = ev.card_markdown(card_v1)
(OUT / "report_card_v1.md").write_text(md_card)
print(md_card)

pts = np.linspace(0.02, 0.8, 40)
y, p = test["y"], v1(test["F"])
nb_model = [ev.net_benefit(y, p, t) for t in pts]
nb_all = [ev.net_benefit_treat_all(y.mean(), t) for t in pts]
plt.plot(pts, nb_model, label="model v1"); plt.plot(pts, nb_all, label="act on everyone"); plt.axhline(0, c="k", lw=1, label="act on no one")
plt.ylim(-0.1, 0.6); plt.xlabel("threshold probability"); plt.ylabel("net benefit"); plt.legend(); plt.show()

In [ ]:
# Checks
assert card_v1["AUC 95% CI"][0] <= card_v1["AUC"] <= card_v1["AUC 95% CI"][1]
assert card_v1["AUC"] > 0.85
assert np.mean([m > max(a, 0) for m, a, t in zip(nb_model, nb_all, pts) if 0.1 <= t <= 0.6]) > 0.9

## Part D · Silent drift

Model v1 goes live on day 1. Each day it scores 20 recordings. From day 31 a quarter of the patients have sinus
rhythm with frequent early beats, which the RR features mistake for AF (m37). Nobody labels the deployment data,
so we monitor what we have: the day's mean score, standardized against the validation period, with a CUSUM alarm
(Page 1954; eq-m45-cusum, allowance $k$ = 0.5 and threshold $h$ = 5 standard deviations, illustrative).

In [ ]:
old = cohort(600, 1, ("sinus", "sinus", "sinus", "af"), 6)
new = cohort(600, 1, ("sinus", "ectopy", "sinus", "af"), 7)
p_old, p_new = v1(old["F"]), v1(new["F"])
rng = np.random.default_rng(5)
per_day, days, change = 20, 60, 30
base = np.array([rng.choice(p_old, per_day).mean() for _ in range(300)])      # validation period
mu0, sd0 = base.mean(), base.std()
daily, picks = [], []
for d in range(days):
    src = old if d < change else new
    idx = rng.choice(len(src["y"]), per_day)
    picks.append((d >= change, idx))
    daily.append(((p_old if d < change else p_new)[idx]).mean())
z = (np.array(daily) - mu0) / sd0
S, alarm = ev.cusum(z, 0.0, 0.5, 5.0)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
a1.plot(np.arange(1, days + 1), z, "o", ms=3); a1.axvline(change + 0.5, c="gray", ls="--"); a1.set_ylabel("daily mean score (SD)")
a2.plot(np.arange(1, days + 1), S); a2.axhline(5.0, c="r", ls="--")
if alarm is not None:
    a2.axvline(alarm + 1, c="g")
a2.set_xlabel("day"); a2.set_ylabel("CUSUM S"); plt.show()
se_old, sp_old = ev.sens_spec(old["y"], p_old, 0.5)
se_new, sp_new = ev.sens_spec(new["y"], p_new, 0.5)
print(f"Alarm on day {alarm + 1}, {alarm + 1 - change} days after the change." if alarm is not None else "No alarm.")
print(f"Before the change: sensitivity {se_old:.2f}, specificity {sp_old:.2f}. After: sensitivity {se_new:.2f}, "
      f"specificity {sp_new:.2f} (in patients with early beats {np.mean(p_new[(new['kind'] == 'ectopy') & (new['y'] == 0)] < 0.5):.2f}).")

In [ ]:
# Checks
assert alarm is not None and change <= alarm < change + 10        # caught within ten days, no false alarm before
assert abs(se_new - se_old) < 0.08 and sp_new < sp_old - 0.15      # sensitivity looks fine; specificity does not

A monitor that tracked only sensitivity on confirmed AF cases would have seen nothing: the drift is all in the
false positives. The score monitor needs no labels, but it only says *something changed*; finding out what needs a
labelled sample (here, the patients with early beats).

## Part E · Change control with criteria fixed in advance

Before any update, write down what kind of change is planned and what it must achieve. Our plan (illustrative):
**retrain the same model, with the same two inputs, on data that include the new population, and re-tune the
threshold if needed.** Acceptance criteria, judged on a locked test set from the new population:

In [ ]:
criteria = {
    "discrimination: AUC lower 95% bound >= 0.85": ("AUC 95% CI", ">=", 0.85),
    "sensitivity lower 95% bound >= 0.80": ("sensitivity 95% CI", ">=", 0.80),
    "specificity lower 95% bound >= 0.80": ("specificity 95% CI", ">=", 0.80),
    "calibration slope between 0.7 and 1.3": ("calibration slope", "between", (0.7, 1.3)),
}
sub_criterion = 0.80     # and specificity >= 0.80 in patients with early beats

locked = cohort(300, 2, ("sinus", "ectopy", "af"), 3)       # locked test set, new population
update = cohort(300, 2, ("sinus", "ectopy", "af"), 4)       # new training data
tune = cohort(300, 2, ("sinus", "ectopy", "af"), 8)         # tuning data for the threshold

v2 = ev.fit_detector(np.r_[train["F"], update["F"]], np.r_[train["y"], update["y"]])
p_tune = v2(tune["F"])
thr_v2 = float(np.quantile(p_tune[tune["y"] == 1], 1 - 0.92))   # threshold for 92% sensitivity on tuning data
v3 = ev.fit_detector(np.r_[train["G"], update["G"]], np.r_[train["y"], update["y"]])   # adds a third input

candidates = [
    ("v1 unchanged", v1(locked["F"]), 0.5, "-"),
    ("v2 retrained", v2(locked["F"]), 0.5, "inside the plan"),
    (f"v2 retrained, threshold {thr_v2:.2f}", v2(locked["F"]), thr_v2, "inside the plan"),
    ("v3 new input (regular-beat share)", v3(locked["G"]), 0.5, "OUTSIDE the plan"),
]
results = []
for name, p, thr, scope in candidates:
    card = ev.report_card(name, locked["y"], p, locked["pid"], thr, subgroups=locked["kind"], rng=np.random.default_rng(0))
    res, ok = ev.pccp_check(card, criteria)
    sub = card["specificity by subgroup"]["ectopy"]
    ok = ok and sub >= sub_criterion
    results.append((name, scope, ok, res, sub))
    print(f"\n{name}  ({scope})  ->  {'PASS' if ok else 'FAIL'}")
    for crit, (obs, passed) in res.items():
        print(f"   {'ok ' if passed else 'NO '} {crit}: {obs:.3f}")
    print(f"   {'ok ' if sub >= sub_criterion else 'NO '} specificity with early beats >= {sub_criterion}: {sub:.3f}")

In [ ]:
# Checks
verdict = {name: ok for name, _, ok, _, _ in results}
assert not verdict["v1 unchanged"]
assert not any(ok for name, scope, ok, _, _ in results if scope == "inside the plan")
assert verdict["v3 new input (regular-beat share)"]

Read the result carefully. No update inside the plan passes: retraining trades sensitivity for specificity, and
re-tuning the threshold trades it back. The change that works adds an input, which the plan did not describe, so
it is not covered by it and would need its own review (m36). Writing the plan and its criteria before seeing the
results is what makes this a test rather than a search for a passing number.

## Part F · Real recordings: train on one database, monitor on another

Model v1 is trained on single-lead recordings from the PhysioNet/CinC Challenge 2017 (Clifford et al. 2017; Open
Data Commons Attribution v1.0) and then "deployed" on lead I of PTB-XL 1.0.3, fold 10 (Wagner et al. 2020; Creative
Commons Attribution 4.0), ten recordings a day in random order. Both downloads are checked against the published
SHA-256 lists (pinned in `data/registry.json`); without internet access this part is skipped. As in m37, the two
databases differ in device, lead and population, so this is an external validation and a monitoring exercise
in one. The intervals here resample recordings, because the loaders do not return patient identifiers.

In [ ]:
from scipy.signal import resample_poly

from epml.data import DataUnavailable, load_challenge2017, load_ptbxl_lead

try:
    sig, lab, _ = load_challenge2017(n_per_class=150, classes=("N", "A"))
    ext, ext_lab, ext_ids, _ = load_ptbxl_lead(n_per_class=60, codes=("SR", "AFIB"), fold=10)
    REAL = True
except DataUnavailable as err:
    print("Part F skipped:", err)
    REAL = False

In [ ]:
if REAL:
    def rr_rows(signals):
        peaks = [ai.detect_r_peaks(np.asarray(x, float), FS) for x in signals]
        return ev.rr_feature_rows(signals, peaks, FS)

    xs = [resample_poly(np.asarray(x, float), 1, 3)[:1000] for x in sig]
    keep = [i for i, x in enumerate(xs) if len(x) == 1000]
    Fc = rr_rows([xs[i] for i in keep]); yc = np.array([lab[i] == "A" for i in keep], int)
    Fp = rr_rows([np.asarray(x)[:1000] for x in ext]); yp = np.array([l == "AFIB" for l in ext_lab], int)
    r = np.random.default_rng(7)
    hold = r.random(len(yc)) < 0.3
    v1_real = ev.fit_detector(Fc[~hold], yc[~hold])
    p_hold, p_ext = v1_real(Fc[hold]), v1_real(Fp)
    base = np.array([r.choice(p_hold, 10).mean() for _ in range(300)])
    order = r.permutation(len(yp))
    daily_real = [p_ext[order[i:i + 10]].mean() for i in range(0, len(order) - 9, 10)]
    S_real, alarm_real = ev.cusum((np.array(daily_real) - base.mean()) / base.std(), 0.0, 0.5, 5.0)
    card_int = ev.report_card("v1, Challenge 2017 hold-out", yc[hold], p_hold, np.arange(hold.sum()), 0.5, rng=r)
    card_ext = ev.report_card("v1, PTB-XL fold 10", yp, p_ext, np.array(ext_ids), 0.5, rng=r)
    for c in (card_int, card_ext):
        print(f"{c['model']:30s} AUC {c['AUC']:.3f} ({c['AUC 95% CI'][0]:.3f}-{c['AUC 95% CI'][1]:.3f}), "
              f"sensitivity {c['sensitivity']:.2f}, specificity {c['specificity']:.2f}, calibration slope {c['calibration slope']:.2f}")
    print(f"{len(daily_real)} deployment days; CUSUM alarm: {'day ' + str(alarm_real + 1) if alarm_real is not None else 'none'}")

In [ ]:
# Checks
if REAL:
    assert 0 <= card_ext["AUC"] <= 1 and len(daily_real) >= 10
    assert np.all(np.isfinite(S_real))

## Where to go next

- In part B, what sample size does a study of an AF detector need if only 2% of screened patients have AF?
- In part D, lower $h$ to 2. How many false alarms appear before day 31 over ten repeats of the simulation?
- In part E, write a plan that would have covered v3, and decide in advance which evidence it should require.
- For a real study, fill in the narrative items of TRIPOD+AI and PROBAST+AI for your report card.